In [ ]:
import websockets
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## Arquivos esperados

Antes de rodar o servico no robo, coloque este arquivo diretamente dentro da pasta `Localizacao`:

- `camera_calibration_results.npz`: salvo pelo notebook `camera_calibration.ipynb`.

A mascara e construida automaticamente no servico a partir do mapa de reprojecao, mantendo apenas os pixels que apontam para dentro da imagem original e aplicando uma erosao para afastar as bordas.

In [ ]:
from pathlib import Path

LOCALIZACAO_DIR = Path.cwd().parent
CALIB_PATH = LOCALIZACAO_DIR / 'camera_calibration_results.npz'

print('CALIB_PATH:', CALIB_PATH)
print('existe:', CALIB_PATH.exists())

## Teste direto do detector

Rode esta celula no robo antes de iniciar o servidor. Ela inicializa o mesmo detector usado pelo servico, captura um quadro e confere se a saida tem o formato esperado.

In [ ]:
from picamera2 import Picamera2
import os

from servico_circulos import Detector, CAMERA_SIZE

Picamera2.set_logging(Picamera2.ERROR)
os.environ['LIBCAMERA_LOG_LEVELS'] = '3'

with Picamera2(tuning=os.environ.get('LIBCAMERA_RPI_TUNING_FILE', None)) as camera:
    camera.configure(camera.create_still_configuration(main={'size': CAMERA_SIZE}))
    camera.start()
    detector = Detector(9, 18, camera)
    coordenadas, timestamp = detector.detecta()
    camera.stop()

print('timestamp:', timestamp)
print('coordenadas:', coordenadas)
assert isinstance(coordenadas, list)
for p in coordenadas:
    assert len(p) == 2

## Teste pelo websocket

Em um terminal do robo, inicie o servico com:

```bash
python3 servico_circulos.py -p 8086
```

In [ ]:
endereco_robo="localhost" # Substitua pelo endereco do robo se necessario
porta_servico="8086"
async with websockets.connect(f"ws://{endereco_robo}:{porta_servico}/wsctrl") as websocket:
    await websocket.send("ack")
    res = json.loads(await  websocket.recv())

assert 'timestamp' in res
assert 'coordenadas' in res
print(res)

In [ ]:
fig, ax = plt.subplots()
ax.set_aspect('equal')
ax.set_xlim(0,400)
ax.set_ylim(-320,320)
for x, y in res["coordenadas"]:
    c = mpatches.Circle((x,y), radius=12)
    ax.add_patch(c)